In [1]:
# %% [markdown]
# # 1. Ingesta (Capa Bronce)
# Convertimos CSV crudo a formato Delta Lake.

# %%
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date
# Para control de la ingesta
from pyspark.sql.functions import current_timestamp, input_file_name
from delta import *
import os

# URL del Master (definida en docker-compose)
master_url = "spark://spark-master:7077"

# Configuración y Añadimos Delta Lake
# Despues de master_url
builder = SparkSession.builder \
    .appName("Lab_SECOP_Bronze") \
    .master(master_url) \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.0.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.executor.memory", "1g") 

spark = configure_spark_with_delta_pip(builder).getOrCreate()

# %%
# LECTURA CSV
print("Leyendo CSV crudo...")
df_raw = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("delimiter", ",") \
    .option("inferSchema", "true") \
    .load("../data/SECOP_II_Contratos_Electronicos.csv")




:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-391978b0-5fe5-4287-b0ed-c195d76fd12c;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.0.0 in central
	found io.delta#delta-storage;3.0.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 143ms :: artifacts dl 5ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.0.0 from central in [default]
	io.delta#delta-storage;3.0.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   

Leyendo CSV crudo...


In [2]:
df_raw.show()

+--------------+-----------+------------+------+------+-----------------------+----------------+-------------------------+---------------------------------------+--------------+----------------------------+-------------------------+-----------+-----------+------------+
|       Entidad|Nit Entidad|Departamento|Ciudad|Estado|Descripcion del Proceso|Tipo de Contrato|Modalidad de Contratacion|Justificacion Modalidad de Contratacion|Fecha de Firma|Fecha de Inicio del Contrato|Fecha de Fin del Contrato|Precio Base|Valor Total|Valor Pagado|
+--------------+-----------+------------+------+------+-----------------------+----------------+-------------------------+---------------------------------------+--------------+----------------------------+-------------------------+-----------+-----------+------------+
|   ALCALDIA 90|       NULL|     BOLIVAR|  NULL|  NULL|                   NULL|            NULL|                     NULL|                                   NULL|    2023-02-28|             

In [3]:
# Normalización técnica de nombres de columnas
# (REQUERIDO por Delta Lake)
nuevas_columnas = [
    c.strip()
     .replace(" ", "_")
     .replace("(", "")
     .replace(")", "")
     .lower()
    for c in df_raw.columns
]
# Aplicamos los nuevos nombres al DataFrame
df_raw = df_raw.toDF(*nuevas_columnas)
print("Nuevas columnas:", df_raw.columns)

# ESCRITURA BRONCE (Delta)
# Agregar variables de control de la ingesta

df_bronze = (
    df_raw
    .withColumn("fecha_ingesta", current_timestamp())
    .withColumn("archivo_origen", input_file_name())
)

# --------------------------------------------
# Escritura en capa Bronze (Delta Lake)
# --------------------------------------------

print("Escribiendo datos en capa Bronze...")
output_path = "/app/data/lakehouse/bronze/secop"

(
    df_bronze.write
    .format("delta")
    .mode("append")   # No borra nada y mantiene el historial delta
    .save(output_path)
)

print(f"Ingesta Bronze completada. Registros procesados: {df_bronze.count()}")


Nuevas columnas: ['entidad', 'nit_entidad', 'departamento', 'ciudad', 'estado', 'descripcion_del_proceso', 'tipo_de_contrato', 'modalidad_de_contratacion', 'justificacion_modalidad_de_contratacion', 'fecha_de_firma', 'fecha_de_inicio_del_contrato', 'fecha_de_fin_del_contrato', 'precio_base', 'valor_total', 'valor_pagado']
Escribiendo datos en capa Bronze...


26/02/01 07:31:44 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/02/01 07:31:46 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
                                                                                

Ingesta Bronze completada. Registros procesados: 1000


In [4]:
# Validación Delta Lake (historial)
print("Historial Delta - Bronze:")
spark.sql(
    f"DESCRIBE HISTORY delta.`{output_path}`"
).show(truncate=False)

Historial Delta - Bronze:


+-------+-----------------------+------+--------+---------+-----------------------------------+----+--------+---------+-----------+--------------+-------------+---------------------------------------------------------------+------------+-----------------------------------+
|version|timestamp              |userId|userName|operation|operationParameters                |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                               |userMetadata|engineInfo                         |
+-------+-----------------------+------+--------+---------+-----------------------------------+----+--------+---------+-----------+--------------+-------------+---------------------------------------------------------------+------------+-----------------------------------+
|0      |2026-02-01 07:31:43.214|NULL  |NULL    |WRITE    |{mode -> Append, partitionBy -> []}|NULL|NULL    |NULL     |NULL       |Serializable  |true         |{numFiles -> 1, nu